# ۴ · رابط خط فرمان (CLI)

نسخه‌ی قبلی به Colab **قفل** بود:

```python
from google.colab import files          # فقط Colab
from google.colab.patches import cv2_imshow
target_folder = "/content/my_gallery"   # مسیر ثابت
choice = input("گزینه را انتخاب کنید: ")  # فقط تعاملی
```

یعنی نه روی سرور اجرا می‌شد، نه در اسکریپت، نه در cron، نه در CI.

این نوت‌بوک یک **فایل واقعی `vie_cli.py`** روی دیسک می‌سازد که هر جا
اجرا می‌شود — ویندوز، لینوکس، سرور، و همچنان Colab.

---

## چرا نوت‌بوک برای CLI؟

که هم فایل قابل‌اجرا را داشته باشی، هم توضیحش را. سلول بعدی فایل را
می‌نویسد؛ سلول‌های بعدتر نشان می‌دهند چطور استفاده شود.

## 📝 ساخت فایل CLI

In [ ]:
cli_source = r'''#!/usr/bin/env python3
"""
vie — Visual Intelligence Engine CLI

    python vie_cli.py index
    python vie_cli.py search face   --image query.jpg
    python vie_cli.py search animal --query "red panda"
    python vie_cli.py search food   --query "French fries"
    python vie_cli.py search text   --query "a man with a blue backpack"

هیچ وابستگی‌ای به Colab ندارد. همه‌ی مسیرها از config می‌آیند.
"""
from __future__ import annotations

import argparse
import json
import logging
import sys
from pathlib import Path

log = logging.getLogger("vie")


# ═══════════════════════════════════════════════════════════════
# تنظیمات — تک منبع حقیقت
# ═══════════════════════════════════════════════════════════════
DEFAULT_CONFIG = {
    "gallery": "data/gallery",
    "database": "data/index.db",
    "device": "auto",
    "half_precision": True,
    "face":   {"match_threshold": 0.46},
    "animal": {"gate_confidence": 0.30, "search_confidence": 0.40,
               "match_threshold": 0.55, "min_crop_px": 40, "padding_ratio": 0.15},
    "food":   {"gate_classes": list(range(39, 56)),
               "search_classes": list(range(45, 56)),
               "gate_confidence": 0.15, "search_confidence": 0.25,
               "match_threshold": 0.60, "min_crop_px": 50, "padding_px": 20,
               "face_overlap_reject": 0.50},
    "text":   {"top_k": 5},
}


def load_config(path: str | None) -> dict:
    cfg = json.loads(json.dumps(DEFAULT_CONFIG))   # کپی عمیق
    if path:
        import yaml
        with open(path, encoding="utf-8") as f:
            user = yaml.safe_load(f) or {}
        for k, v in user.items():
            if isinstance(v, dict) and isinstance(cfg.get(k), dict):
                cfg[k].update(v)
            else:
                cfg[k] = v
    validate_config(cfg)
    return cfg


def validate_config(cfg: dict) -> None:
    """قواعدی که یک کلاس کامل از باگ‌ها را غیرممکن می‌کنند.

    هر دو مورد زیر باگ واقعی در نسخه اصلی بودند.
    """
    problems = []

    # گیت یک فیلتر سخت است: تصویری که رد کند، جستجو هرگز نمی‌بیندش
    for sec in ("animal", "food"):
        if cfg[sec]["gate_confidence"] > cfg[sec]["search_confidence"]:
            problems.append(
                f"{sec}: گیت ({cfg[sec]['gate_confidence']}) سخت‌گیرتر از "
                f"جستجو ({cfg[sec]['search_confidence']}) است")

    # جستجوی کلاسی که ایندکس نشده = بی‌فایده، و در مورد ۵۶ (صندلی) مضر
    stray = set(cfg["food"]["search_classes"]) - set(cfg["food"]["gate_classes"])
    if stray:
        problems.append(f"food: کلاس‌های {sorted(stray)} جستجو می‌شوند ولی ایندکس نشده‌اند")

    if 56 in cfg["food"]["gate_classes"] or 56 in cfg["food"]["search_classes"]:
        problems.append("food: کلاس COCO 56 صندلی است، غذا نیست")

    if problems:
        raise ValueError("تنظیمات نامعتبر:\n  - " + "\n  - ".join(problems))


# ═══════════════════════════════════════════════════════════════
# آرگومان‌ها
# ═══════════════════════════════════════════════════════════════
def build_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(prog="vie", description="Visual Intelligence Engine")
    p.add_argument("--config", default=None, help="فایل YAML تنظیمات")
    p.add_argument("--log-level", default="INFO",
                   choices=["DEBUG", "INFO", "WARNING", "ERROR"])
    p.add_argument("--json", action="store_true", help="خروجی JSON")

    sub = p.add_subparsers(dest="command", required=True)

    idx = sub.add_parser("index", help="ساخت یا به‌روزرسانی ایندکس")
    idx.add_argument("--workers", type=int, default=8)
    idx.add_argument("--no-prune", action="store_true")

    search = sub.add_parser("search", help="جستجو در ایندکس")
    kinds = search.add_subparsers(dest="kind", required=True)

    f = kinds.add_parser("face", help="پیداکردن یک فرد با عکس")
    f.add_argument("--image", required=True, type=Path)

    for name, helptext in (("animal", "جستجوی گونه جانوری"),
                           ("food", "جستجوی غذا"),
                           ("text", "جستجوی متنی آزاد")):
        n = kinds.add_parser(name, help=helptext)
        n.add_argument("--query", required=True)
        n.add_argument("--top-k", type=int, default=None)

    return p


# ═══════════════════════════════════════════════════════════════
# خروجی
# ═══════════════════════════════════════════════════════════════
def render(matches, as_json: bool) -> None:
    if as_json:
        print(json.dumps([{
            "file_name": m["file_name"],
            "file_path": m["file_path"],
            "score": round(m["score"], 4),
            "box": m.get("box"),
        } for m in matches], indent=2, ensure_ascii=False))
        return

    if not matches:
        print("نتیجه‌ای پیدا نشد")
        return
    width = max(len(m["file_name"]) for m in matches)
    for rank, m in enumerate(matches, 1):
        box = f"  box={m['box']}" if m.get("box") else ""
        print(f"[{rank:>2}] {m['file_name']:<{width}}  {m['score']*100:6.2f}%{box}")


# ═══════════════════════════════════════════════════════════════
# main
# ═══════════════════════════════════════════════════════════════
def main(argv=None) -> int:
    args = build_parser().parse_args(argv)
    logging.basicConfig(
        level=getattr(logging, args.log_level),
        format="%(asctime)s | %(levelname)-7s | %(message)s",
        datefmt="%H:%M:%S",
        stream=sys.stderr,
    )

    try:
        cfg = load_config(args.config)
    except (ValueError, FileNotFoundError) as exc:
        log.error("%s", exc)
        return 2

    # ایمپورت‌های سنگین عمداً اینجا هستند تا --help فوری باشد
    if args.command == "index":
        from vie_engine import build_index
        stats = build_index(cfg, workers=args.workers, prune=not args.no_prune)
        print(stats)
        return 0

    if args.command == "search":
        from vie_engine import run_search
        db = Path(cfg["database"])
        if not db.exists():
            log.error("ایندکسی در %s نیست — اول 'index' را اجرا کن", db)
            return 2
        try:
            matches = run_search(args.kind, cfg, query=getattr(args, "query", None),
                                 image=getattr(args, "image", None),
                                 top_k=getattr(args, "top_k", None))
        except (FileNotFoundError, ValueError) as exc:
            log.error("%s", exc)
            return 1
        render(matches, args.json)
        return 0 if matches else 1

    return 2


if __name__ == "__main__":
    sys.exit(main())
'''

from pathlib import Path
Path("vie_cli.py").write_text(cli_source, encoding="utf-8")
print("✅ vie_cli.py نوشته شد -", len(cli_source), "بایت")

## ✅ تست: آیا آرگومان‌ها درست پارس می‌شوند؟

In [ ]:
import subprocess
import sys

print(subprocess.run([sys.executable, "vie_cli.py", "--help"],
                     capture_output=True, text=True).stdout)

In [ ]:
# تست پارس بدون اجرای مدل‌ها
import importlib.util

spec = importlib.util.spec_from_file_location("vie_cli", "vie_cli.py")
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

p = mod.build_parser()
for argv in (["index"],
             ["search", "animal", "--query", "red panda"],
             ["search", "face", "--image", "q.jpg"],
             ["--json", "search", "text", "--query", "sunset", "--top-k", "3"]):
    a = p.parse_args(argv)
    print(f"{str(argv):<62} → command={a.command} kind={getattr(a,'kind',None)}")

print()
# ✅ اعتبارسنجی تنظیمات باید تنظیمات پیش‌فرض را قبول کند
mod.validate_config(mod.load_config(None))
print("✅ تنظیمات پیش‌فرض معتبر است")

## 🧪 تست: آیا اعتبارسنجی جلوی باگ‌های قدیمی را می‌گیرد؟

اینجا عمداً تنظیمات را به همان حالت‌های باگ‌دار نسخه اصلی برمی‌گردانیم
و انتظار داریم **رد شود**.

In [ ]:
import copy

# ═══ باگ ۱: گیت ۰.۴۰، جستجو ۰.۳۰ (وارونه، مثل نسخه اصلی) ═══
bad = copy.deepcopy(mod.DEFAULT_CONFIG)
bad["animal"]["gate_confidence"] = 0.40
bad["animal"]["search_confidence"] = 0.30
try:
    mod.validate_config(bad)
    print("❌ باید رد می‌شد!")
except ValueError as e:
    print("✅ گیت وارونه رد شد:")
    print("  ", str(e).splitlines()[1].strip())

# ═══ باگ ۲: کلاس ۵۶ (صندلی) در جستجو، مثل نسخه اصلی ═══
bad = copy.deepcopy(mod.DEFAULT_CONFIG)
bad["food"]["search_classes"] = list(range(45, 57))     # شامل ۵۶
try:
    mod.validate_config(bad)
    print("❌ باید رد می‌شد!")
except ValueError as e:
    print("\n✅ صندلی رد شد:")
    for line in str(e).splitlines()[1:]:
        print("  ", line.strip())

## 🚀 نحوه‌ی استفاده

```bash
# ساخت ایندکس
python vie_cli.py index --workers 8

# جستجوها
python vie_cli.py search face   --image query.jpg
python vie_cli.py search animal --query "red panda"
python vie_cli.py search food   --query "French fries"
python vie_cli.py search text   --query "a man with a blue backpack"

# خروجی JSON برای اسکریپت‌نویسی
python vie_cli.py --json search text --query "sunset" --top-k 10

# تنظیمات دلخواه
python vie_cli.py --config my.yaml search animal --query zebra
```

**کد خروج:** `0` موفق · `1` نتیجه‌ای نبود · `2` خطای تنظیمات یا فایل

این یعنی در اسکریپت قابل استفاده است:

```bash
if python vie_cli.py search animal --query cat > /dev/null; then
    echo "گربه پیدا شد"
fi
```

---

## 📋 خلاصه

| قبلاً | الان |
|:--|:--|
| `from google.colab import files` | بدون وابستگی به Colab |
| `cv2_imshow` (فقط Colab) | خروجی متنی یا JSON |
| `/content/my_gallery` ثابت | مسیر از config |
| `input()` — فقط تعاملی | آرگومان خط فرمان |
| بدون کد خروج | `0/1/2` برای اسکریپت |
| ثابت‌ها پخش در کد | یک منبع + اعتبارسنجی |